# 3. Modeling

In [1]:
import pandas as pd

import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeRegressor

## 3.1. Оптимизация препроцессора

**Загрузим данные для обучения**

In [3]:
df = pd.read_csv('E:/ML/housing-prices-ml/data/raw/train.csv')
y = np.log1p(df['SalePrice'])
X = df.drop(columns=['SalePrice', 'Id'])

**Разделим данные на обучающую и тестовыую выборки**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    train_size=0.8,
    shuffle=True,
    random_state=46)

Разделим признаки на числовые и катгориальные для корректного построения пайплайна обучения


In [5]:
num_features = X.select_dtypes('number').columns.to_list()
cat_features = X.select_dtypes('str').columns.to_list()

Преобразования по результатам EDA

In [6]:
num_features.remove('MSSubClass')
cat_features.append('MSSubClass')

**Рассмотрим влияние гипотез, выдвинутных на этапе EDA**

In [7]:
absence_features = [
    'PoolQC',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
    'GarageFinish',
    'GarageType',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'Alley',
    'Fence',
    'MiscFeature',
    'MasVnrType'
]

regular_cat_features = []

for feature in cat_features:
    if feature not in absence_features:
        regular_cat_features.append(feature)

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

absence_pipline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

regular_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, regular_cat_features),
    ('num', num_pipeline, num_features)
])


Проверим что будет, если числовые признаки с малым количеством числовых значений представить категориальными

In [ ]:
for feature in ['MoSold', 'YrSold', 'BsmtHalfBath', 'HalfBath', 'BsmtFullBath', 'FullBath', 'Fireplaces', 'KitchenAbvGr', 'GarageCars', 'BedroomAbvGr', 'TotRmsAbvGrd']:
    for i in range(2):
        test_num_features = num_features.copy()
        test_regular_cat_features = regular_cat_features.copy()
        test_absence_features = absence_features.copy()

        test_num_features.remove(feature)
        if i == 0:
            test_regular_cat_features.append(feature)
        else:
            test_absence_features.append(feature)

        preprocessor = ColumnTransformer([
        ('cat_absence', absence_pipline, absence_features),
        ('cat_reg', regular_cat_pipeline, test_regular_cat_features),
        ('num', num_pipeline, test_num_features)
        ])

        model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeRegressor(random_state=46))
        ])

        cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=46
        )

        cv_scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring='neg_root_mean_squared_error'
        )

        cv_rmse = -cv_scores

        if i == 0:
            print(f'Test: {feature} to regular_cat_features')
        else:
            print(f'Test: {feature} to absence_features')

        print(f'CV RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}\n')

Данные манипуляции не улучшили качество

Сильная линейная связь, замеченная у некоторых числовых признаков не должна оказывать влияния на качество модели, основанных на деревьях

Рассмотрим попмжет ли повысить качество удаление сильно несбалансированных категориальных признаков

In [8]:
for feature in ['Street', 'Alley', 'Utilities', 'LandSlope',
                'Condition2', 'RoofMatl', 'Heating', 'CentralAir',
                'Electrical', 'Functional', 'GarageCond', 'PoolQC',
                'MiscFeature', 'Neighborhood', 'Condition1', 'Condition2',
                'HouseStyle', 'Exterior1st', 'Exterior2nd', 'SaleType']:
    
    test_num_features = num_features.copy()
    test_regular_cat_features = regular_cat_features.copy()
    test_absence_features = absence_features.copy()

    if feature in test_regular_cat_features:
        test_regular_cat_features.remove(feature)
    if feature in test_regular_cat_features:
        test_regular_cat_features.remove(feature)

    preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, test_regular_cat_features),
    ('num', num_pipeline, test_num_features)
    ])

    model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=46))
    ])

    cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
    )

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_root_mean_squared_error'
    )

    cv_rmse = -cv_scores
    
    print(f'Test: REMOVE {feature}')
    print(f'CV RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}\n')

Test: REMOVE Street
CV RMSE: 0.21 ± 0.02

Test: REMOVE Alley
CV RMSE: 0.20 ± 0.02

Test: REMOVE Utilities
CV RMSE: 0.21 ± 0.02

Test: REMOVE LandSlope
CV RMSE: 0.20 ± 0.02

Test: REMOVE Condition2
CV RMSE: 0.21 ± 0.01

Test: REMOVE RoofMatl
CV RMSE: 0.21 ± 0.02

Test: REMOVE Heating
CV RMSE: 0.21 ± 0.02

Test: REMOVE CentralAir
CV RMSE: 0.21 ± 0.02

Test: REMOVE Electrical
CV RMSE: 0.20 ± 0.02

Test: REMOVE Functional
CV RMSE: 0.20 ± 0.02

Test: REMOVE GarageCond
CV RMSE: 0.20 ± 0.02

Test: REMOVE PoolQC
CV RMSE: 0.20 ± 0.02

Test: REMOVE MiscFeature
CV RMSE: 0.20 ± 0.02

Test: REMOVE Neighborhood
CV RMSE: 0.21 ± 0.01

Test: REMOVE Condition1
CV RMSE: 0.21 ± 0.01

Test: REMOVE Condition2
CV RMSE: 0.21 ± 0.01

Test: REMOVE HouseStyle
CV RMSE: 0.20 ± 0.02

Test: REMOVE Exterior1st
CV RMSE: 0.20 ± 0.01

Test: REMOVE Exterior2nd
CV RMSE: 0.20 ± 0.01

Test: REMOVE SaleType
CV RMSE: 0.20 ± 0.01



Данные манипуляции не улучшили качество

Некторые катгориальные признаки несут порядковый характер, рассмотрим влияние **OrdinalEncoding** на результат базовой модели

In [20]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# Порядковые категориальные признаки
ordinal_features = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'KitchenQual',
    'HeatingQC',
    'GarageQual',
    'GarageCond',
    'FireplaceQu'
]

# Убираем ordinal-признаки из остальных групп
new_absence_features = [
    feature for feature in absence_features
    if feature not in ordinal_features
]

new_regular_cat_features = [
    feature for feature in regular_cat_features
    if feature not in ordinal_features
]

new_num_features = [
    feature for feature in num_features
    if feature not in ordinal_features
]

# Порядок категорий для каждого ordinal-признака
ordinal_categories = [
    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],  # ExterQual
    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],  # ExterCond
    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],  # BsmtQual
    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],  # KitchenQual
    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],  # HeatingQC
    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],  # GarageQual
    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],  # GarageCond
    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']   # FireplaceQu
]


# Числовые признаки
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Признаки, где отсутствие значения означает отсутствие объекта
absence_pipeline = Pipeline([
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='None'
    )),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Обычные категориальные признаки
regular_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Порядковые категориальные признаки
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='None'
    )),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# Общий preprocessing
preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipeline, new_absence_features),
    ('cat_regular', regular_cat_pipeline, new_regular_cat_features),
    ('ordinal', ordinal_pipeline, ordinal_features),
    ('num', num_pipeline, new_num_features)
])

# Полный pipeline
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=46))
])

# Cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring='neg_root_mean_squared_error'
)

cv_rmse = -cv_scores

print(f'CV RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}')

CV RMSE: 0.21 ± 0.02


Проверено альтернативное кодирование порядковых категориальных признаков с помощью OrdinalEncoder. На используемой модели и схеме кросс-валидации данный вариант показал более низкое качество по сравнению с One-Hot Encoding, поэтому в финальном pipeline сохранено OHE.

## Рассмотрим другие модели

Для анализа рассмотрим следуюшие модели с подбором гиперпараметров:
* Decision Tree 
* Random Forest
* Gradient Boosting
* XGBoost  

Подбор гиперпараметров осуществим методом случайного поиска по сетке

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

Зафиксируем предобработку

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

absence_pipline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

regular_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, regular_cat_features),
    ('num', num_pipeline, num_features)
])

**Произведем подбор гиперпараметров для решающего дерева**

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=46))
    ])

param_grid = {
    "model__max_depth": [3, 5, 7, 10, 15, 20, 30, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8, 12],
    "model__max_features": [1.0, "sqrt", "log2", 0.5, 0.75],
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=100,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=46,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print("Best parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(
    f"CV RMSE: {-search.best_score_:.2f} ± "
    f"{search.cv_results_['std_test_score'][search.best_index_]:.2f}"
)

Подбор гиперпараметров дает результаты котрые лучше бэйзланйа

**Произведем подбор гиперпараметров для случайного леса**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:

model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=46))
    ])

param_grid = {
    "model__max_depth": [3, 5, 7, 10, 15, 20, 30, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8, 12],
    "model__max_features": [1.0, "sqrt", "log2", 0.5, 0.75],
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=100,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=46,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print("Best parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(
    f"CV RMSE: {-search.best_score_:.2f} ± "
    f"{search.cv_results_['std_test_score'][search.best_index_]:.2f}"
)

Результаты улучшились

**Произведем подбор гиперпараметров для градиентного бустинга**

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=46))
    ])

param_grid = {
    "model__n_estimators": [100, 200, 300, 500, 700, 1000],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    "model__max_depth": [2, 3, 4, 5, 6, 8],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__min_samples_leaf": [1, 2, 4, 8, 12],
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=100,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=46,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print("Best parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(
    f"CV RMSE: {-search.best_score_:.2f} ± "
    f"{search.cv_results_['std_test_score'][search.best_index_]:.2f}"
)

**Произведем подбор гиперпараметров для XGBoost**

In [ ]:
from xgboost import XGBRFRegressor

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRFRegressor(random_state=46))
    ])

param_grid = {
    "model__n_estimators": [200, 400, 600, 800, 1000],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15],
    "model__max_depth": [2, 3, 4, 5, 6, 8],
    "model__min_child_weight": [1, 3, 5, 7, 10],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__gamma": [0, 0.01, 0.1, 0.5, 1, 2, 5],
    "model__reg_alpha": [0, 0.001, 0.01, 0.1, 0.5, 1],
    "model__reg_lambda": [0.1, 0.5, 1, 2, 5, 10],
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=1000,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=46,
    n_jobs=-1
)

search.fit(
    X_train,
    y_train
)

print("Best parameters:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

print(
    f"CV RMSE: {-search.best_score_:.2f} ± "
    f"{search.cv_results_['std_test_score'][search.best_index_]:.2f}"
)

## Итоговое предсказание

Загрузим данные для обучения и прогноза

In [ ]:
test_df = pd.read_csv("E:/ML/housing-prices-ml/data/raw/test.csv")
train_df = pd.read_csv("E:/ML/housing-prices-ml/data/raw/train.csv")

In [ ]:
#y_train = np.log1p(train_df['SalePrice'])
y_train = train_df['SalePrice']
X_train = train_df.drop(columns=['SalePrice', 'Id'])

In [ ]:
X_test = test_df

In [ ]:
num_features = X.select_dtypes('number').columns.to_list()
cat_features = X.select_dtypes('str').columns.to_list()

num_features.remove('MSSubClass')

cat_features.append('MSSubClass')

absence_features = [
    'PoolQC',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
    'GarageFinish',
    'GarageType',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'Alley',
    'Fence',
    'MiscFeature',
    'MasVnrType'
]

regular_cat_features = []

for feature in cat_features:
    if feature not in absence_features:
        regular_cat_features.append(feature)

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

absence_pipline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

regular_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, regular_cat_features),
    ('num', num_pipeline, num_features)
])

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(
        min_samples_split=5,
        min_samples_leaf=1,
        max_features=0.75,
        max_depth=None,
        random_state=46))
    ])

In [ ]:
model.fit(X_train, y_train)

In [ ]:
predict = model.predict(X_test)

In [ ]:
result = pd.DataFrame(index=X_test['Id'])
result['SalePrice'] = predict
result

In [ ]:
result.to_csv("E:/ML/housing-prices-ml/data/result/new_submission.csv")